# S6-01: Claude Code 기초 — CLI, CLAUDE.md, Skills, Computer Use API
**Skilljar S6 L01-L04, L07-L08: Anthropic Apps / Claude Code Setup / In Action / MCP / Computer Use**

## 학습 목표
- Claude Code CLI 명령어와 CLAUDE.md 구조를 이해한다
- Skills와 Hooks 설정 파일을 작성할 수 있다
- Computer Use API의 호출 패턴을 이해한다
- 건축공학 프로젝트를 위한 CLAUDE.md를 설계할 수 있다

## 사전 준비
1. Claude Pro 구독 (Claude Code 사용 권한)
2. Node.js 18+ 설치
3. 이 노트북과 같은 폴더에 `.env` 파일 생성:
```
ANTHROPIC_API_KEY="sk-ant-api03-your-key-here"
```

> **참고**: Claude Code 자체는 터미널에서 실행하는 도구이므로, 이 노트북에서는 주로 설정 파일 작성과 Computer Use API 호출을 실습합니다. CLI 실습은 별도 터미널에서 수행합니다.

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Anthropic 클라이언트 생성
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

---
## Exercise 1: CLAUDE.md 작성과 검증

### 배경
`CLAUDE.md`는 Claude Code가 프로젝트를 이해하기 위해 읽는 핵심 파일입니다.
이 파일에 프로젝트 구조, 코딩 규칙, 빌드 명령어를 명시하면,
Claude Code가 일관된 품질의 코드를 생성합니다.

### 문제
건축 구조 설계 검토 프로젝트를 위한 `CLAUDE.md`를 작성하고,
Claude API를 사용하여 CLAUDE.md의 품질을 자동으로 평가하는 함수를 구현하세요.

### 요구사항
1. CLAUDE.md 템플릿을 Python 문자열로 작성하세요
2. Claude API를 호출하여 CLAUDE.md를 평가하는 `evaluate_claude_md()` 함수를 구현하세요
3. 평가 결과를 JSON으로 반환하세요

### 기대 출력
```json
{
  "score": 8,
  "checklist": {
    "project_overview": true,
    "directory_structure": true,
    "coding_conventions": true,
    "build_commands": true,
    "design_standards": true,
    "test_instructions": true
  },
  "suggestions": ["..."]
}
```

In [ ]:
# ===== Exercise 1: CLAUDE.md 작성과 검증 =====

# Step 1: 건축 구조 프로젝트용 CLAUDE.md 템플릿
claude_md_template = """
# CLAUDE.md — KDS 구조 설계 검토 도구

## 프로젝트 개요
KDS 콘크리트구조 설계기준(KDS 14 20)에 따른 RC 구조 부재
설계 적정성을 자동 검토하는 Python CLI 도구.

## 디렉토리 구조
```
structural-review/
├── src/
│   ├── __init__.py
│   ├── beam.py          # RC 보 설계 검토
│   ├── column.py        # RC 기둥 설계 검토
│   ├── materials.py     # 재료 물성 데이터
│   └── report.py        # 검토 보고서 출력
├── tests/
│   ├── test_beam.py
│   ├── test_column.py
│   └── conftest.py
├── data/
│   └── rebar_table.json
├── main.py
├── pyproject.toml
└── CLAUDE.md
```

## 코딩 규칙
- Python 3.11+, 타입 힌트 필수
- KDS 조항 번호를 독스트링에 명시
- 단위: N, mm, MPa (SI 단위)
- 강도감소계수(phi)는 함수 매개변수로 받을 것
- 모든 공개 함수에 pytest 테스트 작성

## 설계 기준
- KDS 14 20 20: 콘크리트구조 부재 설계
- KDS 14 20 22: 전단 및 비틀림
- KDS 41 17 00: 내진설계

## 빌드 & 테스트
```bash
pip install -e \".[dev]\"
pytest tests/ -v --tb=short
black src/ tests/ --check
```
""".strip()

print("CLAUDE.md 템플릿 작성 완료")
print(f"길이: {len(claude_md_template)} 문자")

In [ ]:
import json

# Step 2: CLAUDE.md 품질 평가 함수
def evaluate_claude_md(claude_md_content: str) -> dict:
    """CLAUDE.md의 품질을 평가하여 점수와 체크리스트를 반환

    Args:
        claude_md_content: CLAUDE.md 파일 내용

    Returns:
        dict: {score, checklist, suggestions}
    """
    system = """당신은 Claude Code 프로젝트 설정 전문 리뷰어입니다.

CLAUDE.md 파일을 평가하여 다음 JSON을 반환하세요:
{
  "score": 1-10 (정수),
  "checklist": {
    "project_overview": true/false,
    "directory_structure": true/false,
    "coding_conventions": true/false,
    "build_commands": true/false,
    "design_standards": true/false,
    "test_instructions": true/false
  },
  "suggestions": ["개선 제안 문자열 리스트"]
}

평가 기준:
- project_overview: 프로젝트 목적과 범위가 명확한가
- directory_structure: 디렉토리 구조가 기술되어 있는가
- coding_conventions: 코딩 규칙(언어, 스타일, 타입힌트 등)이 있는가
- build_commands: 빌드/실행 명령어가 있는가
- design_standards: 적용 설계 기준이 명시되어 있는가
- test_instructions: 테스트 실행 방법이 있는가
"""

    messages = [
        {
            "role": "user",
            "content": f"다음 CLAUDE.md를 평가해주세요:\n\n{claude_md_content}"
        },
        {
            "role": "assistant",
            "content": "```json\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


# Step 3: 평가 실행
result = evaluate_claude_md(claude_md_template)
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# Step 4: 검증 함수
def verify_exercise_1(result: dict) -> None:
    """Exercise 1 결과를 검증"""
    # 필수 키 확인
    assert "score" in result, "score 키가 없습니다"
    assert "checklist" in result, "checklist 키가 없습니다"
    assert "suggestions" in result, "suggestions 키가 없습니다"

    # score 범위 확인
    assert 1 <= result["score"] <= 10, f"score가 1-10 범위여야 합니다: {result['score']}"

    # checklist 키 확인
    required_checks = [
        "project_overview", "directory_structure",
        "coding_conventions", "build_commands",
        "design_standards", "test_instructions"
    ]
    for check in required_checks:
        assert check in result["checklist"], f"checklist에 {check}가 없습니다"
        assert isinstance(result["checklist"][check], bool), f"{check}는 bool이어야 합니다"

    # 우리 템플릿은 모든 항목을 포함하므로 대부분 True여야 함
    true_count = sum(1 for v in result["checklist"].values() if v)
    assert true_count >= 4, f"최소 4개 항목이 True여야 합니다: {true_count}개"

    # suggestions는 리스트여야 함
    assert isinstance(result["suggestions"], list), "suggestions는 리스트여야 합니다"

    print(f"PASS: score={result['score']}/10, "
          f"체크리스트 통과={true_count}/6, "
          f"제안 {len(result['suggestions'])}건")


verify_exercise_1(result)

---
## Exercise 2: Skills 정의 파일과 Hooks 설정 생성

### 배경
Claude Code의 **Skills**는 `.claude/skills/` 디렉토리에 SKILL.md 파일로 정의하는 재사용 가능한 워크플로입니다.
**Hooks**는 `.claude/settings.json`에서 이벤트(PreToolUse, PostWrite 등)에 자동 실행되는 명령을 설정합니다.

### 문제
1. 구조 설계 검토 스킬(SKILL.md)을 생성하는 함수를 구현하세요
2. Claude Code 설정 파일(settings.json)을 생성하는 함수를 구현하세요
3. Claude API를 사용하여 생성된 설정의 유효성을 검증하세요

### 기대 출력
- SKILL.md: YAML frontmatter + markdown 본문
- settings.json: hooks와 mcpServers 설정 포함

In [ ]:
# ===== Exercise 2: Skills 정의와 Hooks 설정 =====

# Step 1: SKILL.md 생성 함수
def generate_skill_md(
    skill_name: str,
    description: str,
    steps: list[str],
    input_example: str = "",
    user_invocable: bool = True,
) -> str:
    """Claude Code Skills 정의 파일(SKILL.md)을 생성

    Args:
        skill_name: 스킬 이름
        description: 스킬 설명
        steps: 수행 절차 리스트
        input_example: 입력 형식 예시
        user_invocable: 사용자가 직접 호출 가능 여부

    Returns:
        str: SKILL.md 내용
    """
    # YAML frontmatter
    frontmatter = f"""---
description: {description}
user-invocable: {'true' if user_invocable else 'false'}
---"""

    # Steps 섹션
    steps_md = "\n".join(f"{i+1}. {step}" for i, step in enumerate(steps))

    # 본문 조립
    body = f"""# {skill_name}

## 수행 절차
{steps_md}"""

    if input_example:
        body += f"""\n\n## 입력 형식 예시\n```\n{input_example}\n```"""

    return f"{frontmatter}\n\n{body}"


# 구조 검토 스킬 생성
structural_skill = generate_skill_md(
    skill_name="Structural Review",
    description="KDS 기준에 따른 RC 구조 부재 설계 적정성 검토를 수행합니다",
    steps=[
        "사용자가 제공한 설계 조건을 파악합니다",
        "src/beam.py 또는 src/column.py의 해당 함수를 호출하는 스크립트를 작성합니다",
        "스크립트를 실행하여 결과를 생성합니다",
        "결과를 검토항목별 표로 정리하여 사용자에게 보고합니다",
        "부적합 항목이 있으면 개선안을 제시합니다",
    ],
    input_example="""부재 유형: RC 보
보 폭(b): 350mm
유효깊이(d): 600mm
fck: 27MPa, fy: 400MPa
인장철근: 5-D25
Mu: 380 kN-m, Vu: 250 kN"""
)

print("=== SKILL.md ===")
print(structural_skill)

In [ ]:
# Step 2: settings.json 생성 함수
def generate_settings_json(
    hooks: dict = None,
    mcp_servers: dict = None,
) -> dict:
    """Claude Code 설정 파일(.claude/settings.json)을 생성

    Args:
        hooks: Hook 이벤트별 명령어 설정
        mcp_servers: MCP 서버 설정

    Returns:
        dict: settings.json 내용
    """
    settings = {}

    if hooks:
        settings["hooks"] = hooks

    if mcp_servers:
        settings["mcpServers"] = mcp_servers

    return settings


# 건축 프로젝트용 설정 생성
settings = generate_settings_json(
    hooks={
        "PostWrite": [
            {
                "matcher": "*.py",
                "command": "black --check --quiet $CLAUDE_FILE_PATH || black $CLAUDE_FILE_PATH"
            }
        ],
        "PreToolUse": [
            {
                "matcher": ".env*",
                "command": "echo 'WARNING: .env 파일 접근 차단' && exit 1"
            }
        ]
    },
    mcp_servers={
        "kds-lookup": {
            "command": "python",
            "args": ["-m", "kds_mcp_server"],
            "cwd": "/path/to/kds-server"
        },
        "supabase": {
            "command": "npx",
            "args": ["-y", "@supabase/mcp-server"],
            "env": {
                "SUPABASE_URL": "https://your-project.supabase.co"
            }
        }
    }
)

print("=== settings.json ===")
print(json.dumps(settings, indent=2, ensure_ascii=False))

In [ ]:
# Step 3: Claude API로 설정 유효성 검증
def validate_cc_config(skill_md: str, settings: dict) -> dict:
    """Claude API를 사용하여 Claude Code 설정의 유효성을 검증"""
    system = """당신은 Claude Code 설정 파일 검증 전문가입니다.
SKILL.md와 settings.json의 유효성을 검사하고 결과를 JSON으로 반환하세요.

검증 항목:
1. SKILL.md: frontmatter 형식, description 존재, 수행 절차 존재
2. settings.json: hooks 이벤트명 유효성, MCP 서버 필수 필드

JSON 형식:
{
  "skill_valid": true/false,
  "settings_valid": true/false,
  "issues": ["문제점 리스트"],
  "recommendations": ["개선 권고 리스트"]
}"""

    messages = [
        {
            "role": "user",
            "content": (
                f"다음 Claude Code 설정을 검증해주세요.\n\n"
                f"=== SKILL.md ===\n{skill_md}\n\n"
                f"=== settings.json ===\n{json.dumps(settings, indent=2)}"
            )
        },
        {
            "role": "assistant",
            "content": "```json\n"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


validation = validate_cc_config(structural_skill, settings)
print(json.dumps(validation, indent=2, ensure_ascii=False))

In [ ]:
# Step 4: 검증 함수
def verify_exercise_2(skill_md: str, settings: dict, validation: dict) -> None:
    """Exercise 2 결과를 검증"""
    # SKILL.md 검증
    assert "---" in skill_md, "SKILL.md에 YAML frontmatter가 없습니다"
    assert "description:" in skill_md, "SKILL.md에 description이 없습니다"
    assert "user-invocable:" in skill_md, "SKILL.md에 user-invocable이 없습니다"
    assert "수행 절차" in skill_md or "Steps" in skill_md, "수행 절차 섹션이 없습니다"

    # settings.json 검증
    assert "hooks" in settings, "settings에 hooks가 없습니다"
    assert "mcpServers" in settings, "settings에 mcpServers가 없습니다"

    # hooks 이벤트 확인
    valid_events = {"PreToolUse", "PostToolUse", "PostWrite"}
    for event in settings["hooks"]:
        assert event in valid_events, f"유효하지 않은 hook 이벤트: {event}"

    # MCP 서버 필수 필드 확인
    for name, server in settings["mcpServers"].items():
        assert "command" in server, f"MCP 서버 '{name}'에 command가 없습니다"

    # API 검증 결과 확인
    assert "skill_valid" in validation, "validation에 skill_valid가 없습니다"
    assert "settings_valid" in validation, "validation에 settings_valid가 없습니다"

    print(f"PASS: SKILL.md 유효={validation.get('skill_valid')}, "
          f"settings.json 유효={validation.get('settings_valid')}, "
          f"MCP 서버 {len(settings['mcpServers'])}개, "
          f"Hook 이벤트 {len(settings['hooks'])}개")


verify_exercise_2(structural_skill, settings, validation)

---
## Exercise 3: Computer Use API 패턴 이해

### 배경
Computer Use는 Claude가 스크린샷을 보고 마우스/키보드를 조작하는 기능입니다.
Tool Use와 동일한 메시지 패턴으로 작동합니다.

### 문제
1. Computer Use API 호출 메시지를 구성하는 함수를 작성하세요
2. 응답에서 tool_use 블록을 파싱하는 함수를 작성하세요
3. 시뮬레이션된 Computer Use 루프를 구현하세요

> **참고**: 실제 Computer Use는 Docker/VM 환경에서 실행해야 합니다.
> 이 실습에서는 API 호출 패턴과 응답 파싱을 연습합니다.
> 실제 스크린샷 실행 없이 메시지 구조만 다룹니다.

### 기대 출력
```
Computer Use 요청 구성 완료
도구 정의: computer (1920x1080)
시뮬레이션 루프:
  Step 1: screenshot
  Step 2: left_click (500, 300)
  Step 3: type "KDS 14 20"
  ...
```

In [ ]:
# ===== Exercise 3: Computer Use API 패턴 =====

# Step 1: Computer Use 도구 정의
def create_computer_tool(
    width: int = 1920,
    height: int = 1080
) -> dict:
    """Computer Use 도구 정의를 생성

    Args:
        width: 화면 너비 (px)
        height: 화면 높이 (px)

    Returns:
        dict: Computer Use 도구 정의
    """
    return {
        "type": "computer_20250124",
        "name": "computer",
        "display_width_px": width,
        "display_height_px": height,
    }


computer_tool = create_computer_tool()
print(f"Computer Use 도구 정의:")
print(json.dumps(computer_tool, indent=2))

In [ ]:
# Step 2: Computer Use 액션 시뮬레이터
# (실제 API 호출 없이 메시지 구조만 연습)

def simulate_computer_actions(task: str) -> list[dict]:
    """주어진 작업에 대한 Computer Use 액션 시퀀스를 시뮬레이션

    Args:
        task: 수행할 작업 설명

    Returns:
        list: 시뮬레이션된 액션 시퀀스
    """
    system = """당신은 Computer Use 액션 플래너입니다.
주어진 작업을 수행하기 위한 Computer Use 액션 시퀀스를 JSON으로 생성하세요.

사용 가능한 액션:
- {"action": "screenshot"}
- {"action": "left_click", "coordinate": [x, y]}
- {"action": "right_click", "coordinate": [x, y]}
- {"action": "double_click", "coordinate": [x, y]}
- {"action": "type", "text": "입력할 텍스트"}
- {"action": "key", "key": "Enter"}
- {"action": "scroll", "coordinate": [x, y], "direction": "up/down"}

JSON 배열로 반환하세요. 설명 없이 JSON만 출력하세요."""

    messages = [
        {"role": "user", "content": f"작업: {task}"},
        {"role": "assistant", "content": "```json\n"}
    ]

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        system=system,
        messages=messages,
        stop_sequences=["```"],
        temperature=0.0
    )

    return json.loads(response.content[0].text.strip())


# AutoCAD 작업 시뮬레이션
task = "AutoCAD에서 500x500mm 직사각형 기둥을 그리드 A1 위치에 그리기"
actions = simulate_computer_actions(task)

print(f"작업: {task}")
print(f"\n시뮬레이션된 액션 시퀀스 ({len(actions)}단계):")
for i, action in enumerate(actions, 1):
    print(f"  Step {i}: {json.dumps(action, ensure_ascii=False)}")

In [ ]:
# Step 3: Computer Use API 요청 메시지 구성
def build_computer_use_request(
    task: str,
    screenshot_base64: str = None,
    width: int = 1920,
    height: int = 1080,
) -> dict:
    """Computer Use API 요청 매개변수를 구성

    Args:
        task: 수행할 작업
        screenshot_base64: 현재 화면 스크린샷 (base64)
        width: 화면 너비
        height: 화면 높이

    Returns:
        dict: API 요청 매개변수
    """
    # 메시지 구성
    content = []

    # 스크린샷이 있으면 이미지로 추가
    if screenshot_base64:
        content.append({
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": screenshot_base64,
            },
        })

    content.append({"type": "text", "text": task})

    return {
        "model": "claude-sonnet-4-0",
        "max_tokens": 1024,
        "tools": [create_computer_tool(width, height)],
        "messages": [{"role": "user", "content": content}],
    }


# 요청 구성 (스크린샷 없이 — 구조만 확인)
request_params = build_computer_use_request(
    task="웹 브라우저에서 KDS 콘크리트구조 설계기준을 검색해주세요"
)

print("=== Computer Use API 요청 구조 ===")
# 메시지 내용은 축약하여 출력
display_params = {
    "model": request_params["model"],
    "max_tokens": request_params["max_tokens"],
    "tools": request_params["tools"],
    "messages_count": len(request_params["messages"]),
    "first_message_role": request_params["messages"][0]["role"],
}
print(json.dumps(display_params, indent=2, ensure_ascii=False))

In [ ]:
# Step 4: 검증 함수
def verify_exercise_3(
    computer_tool: dict,
    actions: list,
    request_params: dict,
) -> None:
    """Exercise 3 결과를 검증"""
    # Computer tool 검증
    assert computer_tool["type"] == "computer_20250124", "도구 타입이 올바르지 않습니다"
    assert computer_tool["name"] == "computer", "도구 이름이 올바르지 않습니다"
    assert "display_width_px" in computer_tool, "display_width_px가 없습니다"
    assert "display_height_px" in computer_tool, "display_height_px가 없습니다"

    # 액션 시퀀스 검증
    assert len(actions) >= 2, f"최소 2단계 이상이어야 합니다: {len(actions)}단계"
    valid_actions = {"screenshot", "left_click", "right_click", "double_click",
                     "type", "key", "scroll", "mouse_move"}
    for i, action in enumerate(actions):
        assert "action" in action, f"Step {i+1}: action 키가 없습니다"
        assert action["action"] in valid_actions, (
            f"Step {i+1}: 유효하지 않은 액션 '{action['action']}'"
        )

    # API 요청 구조 검증
    assert "model" in request_params, "request에 model이 없습니다"
    assert "tools" in request_params, "request에 tools가 없습니다"
    assert "messages" in request_params, "request에 messages가 없습니다"
    assert len(request_params["tools"]) >= 1, "최소 1개 도구가 필요합니다"
    assert request_params["tools"][0]["type"] == "computer_20250124", (
        "첫 번째 도구가 computer_20250124이 아닙니다"
    )

    print(f"PASS: 도구 정의 OK, "
          f"시뮬레이션 {len(actions)}단계, "
          f"API 요청 구조 OK")


verify_exercise_3(computer_tool, actions, request_params)

---
## 핵심 정리

| 항목 | 핵심 포인트 |
|------|------------|
| **CLAUDE.md** | 프로젝트 루트에 위치, 구조/규칙/빌드 명령 포함, Claude Code가 자동 로드 |
| **Skills** | `.claude/skills/*/SKILL.md`, YAML frontmatter + 절차 기술, `/skill-name`으로 호출 |
| **Hooks** | `.claude/settings.json`, PreToolUse/PostWrite 이벤트, 자동 린팅/보안 |
| **MCP 연동** | `mcpServers`에 서버 등록, 자연어로 외부 도구 사용 |
| **Computer Use** | `computer_20250124` 도구 타입, Screenshot-Vision-Action 루프 |

### Claude Code CLI 명령어 요약

| 명령어 | 설명 |
|--------|------|
| `claude` | 대화형 모드 시작 |
| `claude "질문"` | 한 번 질문 |
| `claude -p "질문"` | 비대화형 (파이프라인용) |
| `claude /cost` | 비용 확인 |
| `claude /compact` | 대화 이력 압축 |
| `claude /clear` | 대화 초기화 |